<a href="https://colab.research.google.com/github/SalehMousavi/TumorClassifier/blob/main/CompiledNotebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Processing


In [ ]:
!pip install h5py

import os
import h5py
import cv2  # OpenCV for saving images
import numpy as np
import shutil


# Mount Google Drive (if you haven't already)
from google.colab import drive
drive.mount('/content/drive')

## Matlab image conversion


In [ ]:
#the directory containing the .mat files
mat_files_dir = '/content/drive/MyDrive/APS360/Data/brainTumorDataUnprocessedMatlab4/'
output_images_dir = '/content/drive/MyDrive/APS360/Data/ProccessedMatlabData/'

# Create output directory if it doesn't exist
os.makedirs(output_images_dir, exist_ok=True)

# Function to normalize int16 images to uint8 (0-255)
def normalize_image(image):
    image_min = np.min(image)
    image_max = np.max(image)
    image_normalized = (image - image_min) / (image_max - image_min)  # Normalize to 0-1
    image_uint8 = (image_normalized * 255).astype(np.uint8)  # Scale to 0-255
    return image_uint8

# Loop through all .mat files in the directory
files_added = 0
for filename in os.listdir(mat_files_dir):
    if filename.endswith('.mat'):
        # Load the MATLAB file
        mat_file_path = os.path.join(mat_files_dir, filename)
        h5_file = h5py.File(mat_file_path, 'r')

        image_data = h5_file['cjdata']['image'][:]

        # Check if the image is 512x512 and int16
        if image_data.shape == (512, 512) and image_data.dtype == np.int16:
            # Normalize the int16 image to uint8 for saving
            image_normalized = normalize_image(image_data)

            # Save the image as a JPG file
            output_image_path = os.path.join(output_images_dir, f'{filename[:-4]}.jpg')
            cv2.imwrite(output_image_path, image_normalized, [int(cv2.IMWRITE_JPEG_QUALITY), 95])  # Quality setting
            print(f'Saved {output_image_path}')
            files_added += 1
        else:
            print(f'Skipping file {filename}: Image is not 512x512 or not int16')
        h5_file.close()

print(f"Number of files added: {files_added}")

## Image Standardization


In [ ]:
def pad_image(image, target_size=(224, 224)):
    old_size = image.shape[:2]  # height, width
    ratio = min(target_size[0]/old_size[0], target_size[1]/old_size[1])
    new_size = tuple([int(x * ratio) for x in old_size])

    # Resize the image
    resized_image = cv2.resize(image, (new_size[1], new_size[0]))

    # Calculate padding to add
    delta_w = target_size[1] - new_size[1]
    delta_h = target_size[0] - new_size[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    # Pad the image with black pixels (value 0)
    padded_image = cv2.copyMakeBorder(resized_image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    return padded_image

def process_image(image_path, output_dir):
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error loading image: {image_path}")
        return

    # Determine output filename and format
    filename = os.path.basename(image_path)
    output_filename = os.path.splitext(filename)[0] + '_processed.jpg'  # Save as JPG for uniformity
    output_path = os.path.join(output_dir, output_filename)

    # Apply padding or cropping
    processed_image = pad_image(image)

    # Save the processed image
    cv2.imwrite(output_path, processed_image)
    print(f"Processed image saved as: {output_path}")

In [ ]:
img_files_dir = '/content/drive/MyDrive/APS360/Data/KaggleDataSet/Brain Tumor Data Set/Healthy'
output_files_dir = '/content/drive/MyDrive/APS360/Data/KaggleHealthy224x224'
os.makedirs(output_files_dir, exist_ok=True)

In [ ]:
files_added = 0
for filename in os.listdir(img_files_dir):
      img_path = os.path.join(img_files_dir, filename)
      process_image(img_path, output_files_dir)
      files_added += 1
print(f"Number of files added: {files_added}")

In [ ]:
def copy_images(src_folder, dst_folder):
    # Ensure the destination folder exists
    os.makedirs(dst_folder, exist_ok=True)

    # List all files in the source folder
    for filename in os.listdir(src_folder):
        # Construct full file path
        src_path = os.path.join(src_folder, filename)

        # Check if it is a file and has a valid image extension (optional)
        if os.path.isfile(src_path) and filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            dst_path = os.path.join(dst_folder, filename)
            shutil.copy(src_path, dst_path)
            print(f"Copied: {src_path} to {dst_path}")

In [ ]:
src_folder = '/content/drive/MyDrive/APS360/Data/KaggleHealthy224x224'
dst_folder = '/content/drive/MyDrive/APS360/Data/MergedDataSet224x224/NoTumor'
copy_images(src_folder, dst_folder)

src_folder = '/content/drive/MyDrive/APS360/Data/KaggleBrainTumor224x224'
dst_folder = '/content/drive/MyDrive/APS360/Data/MergedDataSet224x224/Tumor'
copy_images(src_folder, dst_folder)

src_folder = '/content/drive/MyDrive/APS360/Data/ProccessedMatlabData224x224'
dst_folder = '/content/drive/MyDrive/APS360/Data/MergedDataSet224x224/Tumor'
copy_images(src_folder, dst_folder)

## Data splitting


## Image Transformation


In [ ]:
master_folder = '/content/drive/MyDrive/APS360/Data'

In [ ]:
import os
from PIL import Image
import random

#4500*0.8 -> tumour-training
#1500*0.8 -> non-tumour-training

transformations = [
    ("RandomRotation", transforms.RandomRotation(degrees=90)),
    ("RandomHorizontalFlip", transforms.RandomHorizontalFlip(p=1.0)),
    ("RandomVerticalFlip", transforms.RandomVerticalFlip(p=1.0)),
    ("RandomAffine", transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))),
    ("Color Jitter", transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1)),
    ("Gaussian Blur", transforms.GaussianBlur(kernel_size=5, sigma=(2, 5)))
]

train_data = torchvision.datasets.ImageFolder(master_folder + '/DatasetSplit/train')

output_folder = os.path.join(master_folder, "Augmented", "NoTumor")
os.makedirs(output_folder, exist_ok=True)

for i, (img, label) in enumerate(train_data):
    if train_data.classes[label] == 'NoTumor':
        original_image_path = os.path.join(output_folder, f"original_{i}.jpg")
        img.save(original_image_path)

        for transform_name, transform in transformations:
            augmented_img = transform(img)

            # Save the augmented image with a specific suffix
            augmented_image_path = os.path.join(output_folder, f"{transform_name}_{i}.jpg")
            augmented_img.save(augmented_image_path)


In [ ]:
output_folder = os.path.join(master_folder, "Augmented", "Tumor")
os.makedirs(output_folder, exist_ok=True)

for i, (img, label) in enumerate(train_data):
    if train_data.classes[label] == 'Tumor':
        original_image_path = os.path.join(output_folder, f"original_{i}.jpg")
        img.save(original_image_path)

        chosen_transforms = random.sample(transformations, 2)

        for j, (transform_name, transform) in enumerate(chosen_transforms, 1):
            augmented_img = transform(img)
            augmented_image_path = os.path.join(output_folder, f"{transform_name}_{i}_variant{j}.jpg")
            augmented_img.save(augmented_image_path)


In [ ]:
import os

# Define the folders
master_folder = "/content/drive/MyDrive/APS360/Data"
tumor_folder = os.path.join(master_folder, "Augmented", "Tumor")
nontumor_folder = os.path.join(master_folder, "Augmented", "NoTumor")

def count_images_in_folder(folder_path):
    image_count = len([f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
    return image_count

# Count images in each folder
tumor_image_count = count_images_in_folder(tumor_folder)
nontumor_image_count = count_images_in_folder(nontumor_folder)

print(f"Number of images in 'Tumor' folder: {tumor_image_count}")
print(f"Number of images in 'NonTumor' folder: {nontumor_image_count}")

# Primary Model


In [ ]:
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import DataLoader, random_split, TensorDataset
import torchvision.transforms as transforms

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## Transfer Learning


In [ ]:
###############################################################################
# Data Loading

def get_training_loader(batch_size):
    # transform used for image loader
    transform = transforms.Compose([transforms.ToTensor()])
    data_set = torchvision.datasets.ImageFolder(
        root='/content/gdrive/MyDrive/APS360/Data/Augmented',
        transform=transform)

    data_set_size = len(data_set)

    classes = data_set.classes
    print("Total training images: ", data_set_size)

    train_loader = DataLoader(data_set, batch_size=batch_size, shuffle=True)

    return train_loader

def get_validation_loader(batch_size):
    # transform used for image loader
    transform = transforms.Compose([transforms.ToTensor()])
    data_set = torchvision.datasets.ImageFolder(
        root='/content/gdrive/MyDrive/APS360/Data/DatasetSplit/val',
        transform=transform)

    data_set_size = len(data_set)

    classes = data_set.classes
    print("Total validation images: ", data_set_size)

    validation_loader = DataLoader(data_set, batch_size=batch_size, shuffle=False)

    return validation_loader

def get_test_loader(batch_size):
    # transform used for image loader
    transform = transforms.Compose([transforms.ToTensor()])
    data_set = torchvision.datasets.ImageFolder(
        root='/content/gdrive/MyDrive/APS360/Data/DatasetSplit/test',
        transform=transform)

    data_set_size = len(data_set)

    classes = data_set.classes
    #print(classes)
    print("Total test images: ", data_set_size)

    test_loader = DataLoader(data_set, batch_size=batch_size, shuffle=False)

    return test_loader

def get_data_loaders(batch_size):
    train_loader = get_training_loader(batch_size)
    validation_loader = get_validation_loader(batch_size)
    test_loader = get_test_loader(batch_size)

    classes = train_loader.dataset.classes
    print(classes)

    return train_loader, validation_loader, test_loader, classes


In [ ]:
import torchvision.models
device = "cuda" if torch.cuda.is_available() else "cpu"

alexnet = torchvision.models.alexnet(pretrained=True)
if device == "cuda":
  alexnet = alexnet.cuda()

print(device)

In [ ]:
# img = ... a PyTorch tensor with shape [N,3,224,224] containing hand images ...
train_loader, val_loader, test_loader, classes = get_data_loaders(batch_size=256)

def compute_alexnet_features(dataloader):
    all_features = []
    all_labels = []

    with torch.no_grad():  # We don't need gradients for feature extraction
        for inputs, labels in dataloader:
            # Move the data to the device (CPU/GPU)
            if device == "cuda":
              inputs = inputs.cuda()
              labels = labels.cuda()

            # Extract features
            features = alexnet.features(inputs)  # Output will have shape [N, 256, 6, 6]

            # Save features and labels
            all_features.append(features.cpu())  # Move back to CPU if on GPU
            all_labels.append(labels)

    # Concatenate all features and labels into one tensor
    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return all_features, all_labels

train_features, train_labels = compute_alexnet_features(train_loader)
val_features, val_labels = compute_alexnet_features(val_loader)
test_features, test_labels = compute_alexnet_features(test_loader)

In [ ]:
save_path = '/content/gdrive/MyDrive/APS360/AlexNetFeatureMaps/'
import os
if not os.path.exists(save_path):
  os.mkdir(save_path)
torch.save(train_features, save_path + 'train_features.pt')
torch.save(val_features, save_path + 'val_features.pt')
torch.save(test_features, save_path + 'test_features.pt')
torch.save(train_labels, save_path + 'train_labels.pt')
torch.save(val_labels, save_path + 'val_labels.pt')
torch.save(test_labels, save_path + 'test_labels.pt')

## Helper Functions

In [ ]:
def train_net(net, train_loader, val_loader, batch_size=64, learning_rate=0.01, num_epochs=30):
    if device == 'cuda':
        net = net.cuda()
    ########################################################################
    # Train a classifier on cats vs dogs
    target_classes = ["tumor", "no_tumor"]
    ########################################################################
    # Fixed PyTorch random seed for reproducible result
    torch.manual_seed(1000)
    ########################################################################
    # Define the Loss function and optimizer
    # The loss function will be Binary Cross Entropy (BCE). In this case we
    # will use the BCEWithLogitsLoss which takes unnormalized output from
    # the neural network and scalar label.
    # Optimizer will be SGD with Momentum.
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=0.9)
    ########################################################################
    # Set up some numpy arrays to store the training/test loss/erruracy
    train_err = np.zeros(num_epochs)
    train_loss = np.zeros(num_epochs)
    val_err = np.zeros(num_epochs)
    val_loss = np.zeros(num_epochs)
    ########################################################################
    # Train the network
    # Loop over the data iterator and sample a new batch of training data
    # Get the output from the network, and optimize our loss function.
    start_time = time.time()
    for epoch in range(num_epochs):  # loop over the dataset multiple times
        total_train_loss = 0.0
        total_train_err = 0.0
        total_epoch = 0
        for i, data in enumerate(train_loader, 0):
            # Get the inputs
            inputs, labels = data
            if device == 'cuda':
                inputs = inputs.cuda()
                labels = labels.cuda()
            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass, backward pass, and optimize
            outputs = net(inputs)
            loss = criterion(outputs, labels.float())
            loss.backward()
            optimizer.step()
            # Calculate the statistics
            corr = (outputs > 0.0).squeeze().long() != labels
            total_train_err += int(corr.sum())
            total_train_loss += loss.item()
            total_epoch += len(labels)
        train_err[epoch] = float(total_train_err) / total_epoch
        train_loss[epoch] = float(total_train_loss) / (i+1)
        val_err[epoch], val_loss[epoch], f1_score, percision, recall = evaluate(net, val_loader, criterion)
        print(("Epoch {}: Train err: {}, Train loss: {} |"+
               "Validation err: {}, Validation loss: {}").format(
                   epoch + 1,
                   train_err[epoch],
                   train_loss[epoch],
                   val_err[epoch],
                   val_loss[epoch]))
        # Save the current model (checkpoint) to a file
        model_path = get_model_name(net.name, batch_size, learning_rate, epoch)
        torch.save(net.state_dict(), model_path)
    print('Finished Training')
    end_time = time.time()
    elapsed_time = end_time - start_time
    print("Total time elapsed: {:.2f} seconds".format(elapsed_time))
    # Write the train/test loss/err into CSV file for plotting later
    epochs = np.arange(1, num_epochs + 1)
    np.savetxt("{}_train_err.csv".format(model_path), train_err)
    np.savetxt("{}_train_loss.csv".format(model_path), train_loss)
    np.savetxt("{}_val_err.csv".format(model_path), val_err)
    np.savetxt("{}_val_loss.csv".format(model_path), val_loss)

def get_model_name(name, batch_size, learning_rate, epoch):
    """ Generate a name for the model consisting of all the hyperparameter values

    Args:
        config: Configuration object containing the hyperparameters
    Returns:
        path: A string with the hyperparameter name and value concatenated
    """
    path = "model_{0}_bs{1}_lr{2}_epoch{3}".format(name,
                                                   batch_size,
                                                   learning_rate,
                                                   epoch)
    return path


def evaluate(net, loader, criterion):
    """ Evaluate the network on the validation set.

     Args:
         net: PyTorch neural network object
         loader: PyTorch data loader for the validation set
         criterion: The loss function
     Returns:
         err: A scalar for the avg classification error over the validation set
         loss: A scalar for the average loss function over the validation set
     """
    if device == 'cuda':
        net = net.cuda()
    total_loss = 0.0
    total_err = 0.0
    total_epoch = 0

    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0

    for i, data in enumerate(loader, 0):
        inputs, labels = data
        if device == 'cuda':
            inputs = inputs.cuda()
            labels = labels.cuda()
        outputs = net(inputs)
        loss = criterion(outputs, labels.float())
        corr = (outputs > 0.0).squeeze().long() != labels
        total_err += int(corr.sum())
        total_loss += loss.item()
        total_epoch += len(labels)


        predictions = (outputs > 0.0).squeeze().long()
        true_positives += int(((predictions == 1) & (labels == 1)).sum().item())
        false_positives += int(((predictions == 1) & (labels == 0)).sum().item())
        false_negatives += int(((predictions == 0) & (labels == 1)).sum().item())


    err = float(total_err) / total_epoch
    loss = float(total_loss) / (i + 1)

    precision = (true_positives / (true_positives + false_positives)) if (true_positives + false_positives) > 0 else 0
    recall = (true_positives / (true_positives + false_negatives)) if (true_positives + false_negatives) > 0 else 0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    return err, loss, f1_score, precision, recall

###############################################################################
# Training Curve
def plot_training_curve(path):
    """ Plots the training curve for a model run, given the csv files
    containing the train/validation error/loss.

    Args:
        path: The base path of the csv files produced during training
    """
    import matplotlib.pyplot as plt
    train_err = np.loadtxt("{}_train_err.csv".format(path))
    val_err = np.loadtxt("{}_val_err.csv".format(path))
    train_loss = np.loadtxt("{}_train_loss.csv".format(path))
    val_loss = np.loadtxt("{}_val_loss.csv".format(path))
    plt.title("Train vs Validation Error")
    n = len(train_err) # number of epochs
    plt.plot(range(1,n+1), train_err, label="Train")
    plt.plot(range(1,n+1), val_err, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend(loc='best')
    plt.show()
    plt.title("Train vs Validation Loss")
    plt.plot(range(1,n+1), train_loss, label="Train")
    plt.plot(range(1,n+1), val_loss, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc='best')
    plt.show()

## Create Dataloaders

In [ ]:
save_path = '/content/gdrive/MyDrive/APS360/AlexNetFeatureMaps/'
train_features = torch.load(save_path + 'train_features.pt')
train_labels = torch.load(save_path + 'train_labels.pt')

train_dataset = TensorDataset(train_features, train_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_features = torch.load(save_path + 'val_features.pt')
val_labels = torch.load(save_path + 'val_labels.pt')

# Create TensorDataset for validation features and labels
val_dataset = TensorDataset(val_features, val_labels)

# Create DataLoader for the validation set
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# features = ... load precomputed alexnet.features(img) ...
test_features = torch.load(save_path + 'test_features.pt')
test_labels = torch.load(save_path + 'test_labels.pt')

# Create TensorDataset for test features and labels
test_dataset = TensorDataset(test_features, test_labels)

# Create DataLoader for the test set
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Classifer Module

In [ ]:
# define a 2-layer artificial neural network
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.name = "classifier"
        self.layer1 = nn.Linear(256 * 6 * 6, 64)
        self.layer2 = nn.Linear(64, 1)
        self.func = nn.ReLU()
    def forward(self, img):
        flattened = img.view(-1, 256 * 6 * 6)
        activation1 = self.layer1(flattened)
        activation1 = self.func(activation1)
        activation2 = self.layer2(activation1)
        return activation2.squeeze(1)

## Training

In [ ]:
classifier = Classifier()
if device == "cuda":
  classifier = classifier.cuda()

train_net(classifier, train_loader, val_loader, batch_size=32, num_epochs=30)

In [ ]:
net = Classifier()
model_path = get_model_name(net.name, batch_size=32, learning_rate=0.01, epoch=14)
state = torch.load(model_path)
net.load_state_dict(state)

if device == "cuda":
  net = net.cuda()

criterion = nn.BCEWithLogitsLoss()
test_acc, test_loss, f1_score, percision, recall = evaluate(net, test_loader, criterion)
print(f"Test Accuracy: {1-test_acc}, F1-Score: {f1_score}, Percision: {percision}, Recall: {recall} ")

In [ ]:
# features = ... load precomputed alexnet.features(img) ...
test_features = torch.load(save_path + 'test_features.pt')
test_labels = torch.load(save_path + 'test_labels.pt')

# Create TensorDataset for test features and labels
test_dataset = TensorDataset(test_features, test_labels)

# Create DataLoader for the test set
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.eval()  # Set model to evaluation mode

# Get the first data point
with torch.no_grad():  # No gradients needed for testing
    for features, labels in test_loader:
        # Select only the first data point in the batch
        if(labels[0] == 1):
          first_feature = features[0].unsqueeze(0).to(device)  # Add batch dimension back
          first_label = labels[0].unsqueeze(0).to(device)       # Optional if you want the label
          output = net(first_feature)
          print("Model output:", output)
          print("True label:", first_label)  # Optional, if label is needed
          break  # Exit after first batch

## Baseline Model

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
import torch.nn as nn
import torch.optim as optim
import time
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, auc, roc_curve
from sklearn.preprocessing import StandardScaler
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC
import torch
import torchvision
import torchvision.transforms as transforms
from google.colab import drive

In [ ]:
# Determine the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Mount Google Drive
drive.mount('/content/gdrive')

# Define dataset paths
train_folder = '/content/gdrive/MyDrive/APS360/Data/Augmented'
val_folder = '/content/gdrive/MyDrive/APS360/Data/DatasetSplit/val'
test_folder = '/content/gdrive/MyDrive/APS360/Data/DatasetSplit/test'

# Load the datasets
train_data = torchvision.datasets.ImageFolder(root=train_folder, transform=transforms.ToTensor())
val_data = torchvision.datasets.ImageFolder(root=val_folder, transform=transforms.ToTensor())
test_data = torchvision.datasets.ImageFolder(root=test_folder, transform=transforms.ToTensor())

import numpy as np
import torch
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class TumourClassifier:
    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.setup_model()

    def setup_model(self):
        # Initialize SVM with probability estimates and a standard scaler
        self.svm = svm.SVC(kernel='linear', probability=True, class_weight='balanced')
        self.scaler = StandardScaler()

    def train_svm(self, train_loader):
        # Iterate through batches
        for batch_idx, (images, labels) in enumerate(train_loader):
            # Flatten and move to CPU
            images_np = images.view(images.size(0), -1).cpu().numpy()
            labels_np = labels.cpu().numpy()

            # Scale the features (fit on first batch, transform on subsequent batches)
            if batch_idx == 0:
                self.scaler.fit(images_np)  # Fit scaler on the first batch
            images_np = self.scaler.transform(images_np)  # Transform for all batches

            # Incremental fitting for SVM
            self.svm.fit(images_np, labels_np)

            print(f"Processed batch {batch_idx + 1}/{len(train_loader)}")

# Example usage
train_folder = '/content/gdrive/MyDrive/APS360/Data/Augmented'
train_data = datasets.ImageFolder(root=train_folder, transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

classifier = TumourClassifier()
classifier.train_svm(train_loader)
print("Training complete.")

In [ ]:
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, auc, roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, auc, roc_curve
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
def predict(self, data_loader):
        predictions = []
        labels = []

        # Iterate through the DataLoader
        for images, lbls in data_loader:
            images_np = images.view(images.size(0), -1).numpy()
            labels.extend(lbls.numpy())  # Collect labels

            # Scale the images using the same scaler
            images_scaled = self.scaler.transform(images_np)
            pred = self.svm.predict(images_scaled)
            predictions.extend(pred)  # Collect predictions

        return np.array(predictions), np.array(labels)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# Get test predictions and performance
test_predictions, test_labels = predict(classifier, test_loader)
test_accuracy = accuracy_score(test_labels, test_predictions)
test_precision = precision_score(test_labels, test_predictions, average='weighted')
test_recall = recall_score(test_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_labels, test_predictions, average='weighted')

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")